SKIP: interactive analysis notebook (no savefig). Re-export manually as needed.

In [1]:
import pandas as pd
import json

df = pd.read_csv("../data/module_time.csv")
df["duration"] = df["end_time"] - df["start_time"]

df[["run", "pipeline", "stage", "node_type", "node_num", "ip", "gpu", "module",
    "duration", "loading_time"]].round(2)

,run,pipeline,stage,node_type,node_num,ip,gpu,module,duration,loading_time
0,0,1,0,g6_12xlarge,1,192.168.0.226,all,apiserver,78.20,78.20
1,0,1,0,g6_12xlarge,1,192.168.0.226,0,tensorstore,23.59,23.59
2,0,1,0,g6_12xlarge,1,192.168.0.226,1,tensorstore,24.04,24.04
3,0,1,0,g6_12xlarge,1,192.168.0.226,2,tensorstore,23.59,23.59
4,0,1,0,g6_12xlarge,1,192.168.0.226,3,tensorstore,24.37,24.37
...,...,...,...,...,...,...,...,...,...,...
281,10,2,2,g5_12xlarge,2,192.168.0.240,0,tensorstore,39.84,39.84
282,10,2,2,g5_12xlarge,2,192.168.0.240,1,tensorstore,39.16,39.16
283,10,2,2,g5_12xlarge,2,192.168.0.240,2,tensorstore,38.43,38.43
284,10,2,2,g5_12xlarge,2,192.168.0.240,3,tensorstore,41.89,41.89


In [2]:
# 각 run(iteration), pipeline별 max tensorstore loading_time
max_ts = (df[df["module"] == "tensorstore"]
          .groupby(["run", "pipeline"])["loading_time"]
          .max()
          .reset_index()
          .rename(columns={"loading_time": "max_ts_loading_time (s)"}))

# 각 run(iteration), pipeline별 apiserver loading_time
api = (df[df["module"] == "apiserver"]
       .groupby(["run", "pipeline"])[["loading_time"]]
       .first()
       .reset_index()
       .rename(columns={"loading_time": "apiserver_loading_time (s)"}))

summary = max_ts.merge(api, on=["run", "pipeline"], how="left")
summary.round(2)

,run,pipeline,max_ts_loading_time (s),apiserver_loading_time (s)
0,0,1,56.08,78.20
1,0,2,65.59,77.85
2,1,1,57.70,60.59
3,1,2,81.45,81.84
4,2,1,52.99,55.81
5,2,2,79.98,77.65
6,3,1,71.03,76.33
7,3,2,77.70,77.77
8,4,1,52.78,56.06
9,4,2,64.63,63.75


In [3]:
cols = ["max_ts_loading_time (s)", "apiserver_loading_time (s)"]

print("=== Per Pipeline ===")
display(summary.groupby("pipeline")[cols].agg(["mean", "std", "min", "max"]).round(2))

print("\n=== Overall ===")
display(summary[cols].agg(["mean", "std", "min", "max"]).round(2))

=== Per Pipeline ===


max_ts_loading_time (s)                      \
                            mean   std    min    max   
pipeline                                               
1                          54.97  5.85  50.90  71.03   
2                          68.72  7.39  60.06  81.45   

         apiserver_loading_time (s)                      
                               mean   std    min    max  
pipeline                                                 
1                             60.79  8.38  55.57  78.20  
2                             68.22  8.90  57.51  81.84


=== Overall ===


,max_ts_loading_time (s),apiserver_loading_time (s)
mean,61.85,64.51
std,9.59,9.25
min,50.90,55.57
max,81.45,81.84


In [4]:
with open("../data/time_to_start.json") as f:
    data = json.load(f)

df_start = pd.DataFrame(data)
df_start = df_start[["instance_type", "iteration",
                      "time_to_fulfillment", "time_to_running", "time_to_ssh_ready"]]
df_start = df_start.rename(columns={
    "time_to_fulfillment": "fulfillment (s)",
    "time_to_running": "running (s)",
    "time_to_ssh_ready": "ssh_ready (s)",
})
df_start.round(2)

,instance_type,iteration,fulfillment (s),running (s),ssh_ready (s)
0,g6.12xlarge,1,3.08,24.12,26.73
1,g6.12xlarge,2,3.09,24.11,27.33
2,g6.12xlarge,3,3.14,20.01,49.46
3,g5.12xlarge,1,3.11,15.84,46.99
4,g5.12xlarge,2,3.12,22.02,46.57
5,g5.12xlarge,3,3.20,18.02,45.57
6,g6e.xlarge,1,3.25,9.73,39.88
7,g6e.xlarge,2,3.12,9.57,40.12
8,g6e.xlarge,3,3.11,9.55,38.61
9,g6.12xlarge,1,3.17,24.18,52.03


In [5]:
metrics = ["fulfillment (s)", "running (s)", "ssh_ready (s)"]

print("=== Per Instance Type ===")
display(df_start.groupby("instance_type")[metrics].agg(["mean", "std", "min", "max"]).round(2))

print("\n=== Overall ===")
display(df_start[metrics].agg(["mean", "std", "min", "max"]).round(2))

=== Per Instance Type ===


fulfillment (s)                   running (s)               \
                         mean   std   min   max        mean   std    min   
instance_type                                                              
g5.12xlarge              3.12  0.04  3.07  3.20       19.64  3.04  15.84   
g6.12xlarge              3.12  0.05  3.06  3.19       21.73  2.73  18.00   
g6e.xlarge               3.56  0.84  3.11  5.25        9.65  0.11   9.55   

                     ssh_ready (s)                       
                 max          mean    std    min    max  
instance_type                                            
g5.12xlarge    24.10         45.39   2.09  41.69  47.14  
g6.12xlarge    24.18         42.07  11.81  26.73  52.03  
g6e.xlarge      9.84         37.20   2.91  32.58  40.12


=== Overall ===


,fulfillment (s),running (s),ssh_ready (s)
mean,3.27,17.01,41.55
std,0.50,5.86,7.54
min,3.06,9.55,26.73
max,5.25,24.18,52.03
